# E18 · Training on all three image sets at once

We now have every EuroSAT patch three times, with the same label:

| Set | Processing | Season | Alone on Kaggle |
|---|---|---|---|
| EuroSAT L1C + `hist_split` | top of atmosphere, haze corrected | original (unknown) | 72.67% (E09a) |
| L2A April–October 2017–2018 | atmospherically corrected | mostly September | 71.97% (E17a) |
| L2A June–August 2017, 2019, 2020 | atmospherically corrected | midsummer | 72.02% (E17b) |

Each set alone is around 72%, and each L2A model raised the ensemble (E15d). E18 trains **one** model on all three: it
sees every place in three versions and has to learn what the land cover is, not what the processing or the season
looks like. We do not know when the Kaggle images were taken, so robustness to season is what we want.

Setup as E09a/E17 (EfficientNet-B0, 12 bands, contrast augmentation, round-3 pseudo-labels). Two adjustments that
follow from the 3× larger training set:

* pseudo-labels used **6×** instead of 2×, so they keep the same share of the training set (about 13%);
* **12 epochs** instead of 20 (each epoch has 3× more images, so this is still more training steps).

Validation, internal test and the normaliser use the midsummer L2A set, the one closest to Kaggle.

**Train or load.** `LOAD_RUNS["E18a"] = None` trains (about 3× an E09a run). Commit the weights if submitted.

## 1 · Setup

In [ ]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.data import CLASSES, PROJECT_ROOT, TEST_BANDS
from src.experiment import (add_pseudo_labels, class_mix, compare_variants, find_run, finish_run, kaggle_monitor,
                            kaggle_score, load_experiment_data, load_saved_model, plot_class_mix,
                            plot_variant_curves, select_view_agree, teacher_predictions, view_agreement)
from src.models import MEDIUM_TRACK_LIMIT, build_model
from src.plot_style import apply_style
from src.training import Normalizer, TrainConfig, fit, load_run, predict_views
from src.utils import describe_device, get_device, seed_everything

apply_style()

# ---- Experiment settings ---------------------------------------------------------------
NOTEBOOK = "E18_all_image_sets"
LOAD_RUNS = {   # saved runs; None = train this variant again
    "E18a": None,
}
REFERENCE = "pl_r3_view_agree_contrast_aug_hist_split_effb0"               # E09a, no AdaBN, 72.67%
R3_TEACHER = "adabn_balanced_pseudo_label_r2_x2_contrast_aug_hist_split_effb0"   # E06c + AdaBN: round-3 labels
CAPS = {"SeaLake": 400}
REPEAT = 6                     # 3x the E09a value (2), because the EuroSAT part is 3x larger
MODEL = "efficientnet_b0"
BANDS = list(TEST_BANDS)
SEED = 42
cfg = TrainConfig(epochs=12, batch_size=128, lr=1e-3, weight_decay=1e-4,
                  label_smoothing=0.1, patience=4, augment=True,
                  photometric={"contrast": (0.9, 1.5), "band_contrast": (0.9, 1.15),
                               "brightness": (0.93, 1.07), "p": 0.8})   # as in E09a, fewer epochs
# -----------------------------------------------------------------------------------------

seed_everything(SEED)
device = get_device()
print("Device:", describe_device(device))
results = {}

## 2 · The three training sets

Only the training split of the L1C and April–October sets is kept; everything else (validation, internal test,
Kaggle, normaliser) comes from the midsummer set. All three use the same split, so no validation or test patch is
trained on in any version.

In [ ]:
parts = {}
for name, corrections, source in [("L1C + hist_split", ["hist_split"], "l1c"),
                                  ("L2A Apr-Oct", [], "l2a")]:
    d = load_experiment_data(BANDS, corrections, device, source=source)
    parts[name] = (d.tr_images, d.tr_labels)
    del d
    if device.type == "mps":
        torch.mps.empty_cache()

data = load_experiment_data(BANDS, [], device, source="l2a_summer")   # L2A Jun-Aug: everything else
print(data.summary())
table, _ = teacher_predictions(R3_TEACHER, data, device)
hard_idx = select_view_agree(table, CAPS, SEED)
summer_images, summer_labels, _ = add_pseudo_labels(data, table, hard_idx, repeat=REPEAT)

tr_images = torch.cat([summer_images] + [p[0] for p in parts.values()])
tr_labels = torch.cat([summer_labels] + [p[1] for p in parts.values()])
del parts, summer_images, summer_labels
pd.Series({"L2A Jun-Aug (+ pseudo-labels)": len(data.tr_images) + REPEAT * len(hard_idx),
           "L1C + hist_split": len(data.tr_images), "L2A Apr-Oct": len(data.tr_images),
           "total": len(tr_images)}).rename("training images (approx.)")

## 3 · E18a · one model, three image sets

In [ ]:
if LOAD_RUNS["E18a"]:
    model, normalizer, history = load_saved_model(LOAD_RUNS["E18a"], device)
    best_state = None
else:
    seed_everything(SEED)
    normalizer = Normalizer.fit(data.X_tr)
    model = build_model(MODEL, bands=BANDS, pretrained=True).to(device)
    history, best_state = fit(model, tr_images, tr_labels, data.va_images, data.va_labels,
                              normalizer, cfg, monitor=kaggle_monitor(data, normalizer))

results["E18a"] = finish_run(model, normalizer, history, best_state, data, variant="E18a", load_run=LOAD_RUNS["E18a"],
    device=device, seed=SEED, param_limit=MEDIUM_TRACK_LIMIT,
    config={"run_name": "all_sets_pl_r3_view_agree_contrast_aug_effb0",
            "description": "one model on EuroSAT L1C + hist_split, L2A Apr-Oct and L2A Jun-Aug (+ round-3 pseudo-labels 6x)",
            "notebook": NOTEBOOK, "variant": "E18a", "model": MODEL, "bands": BANDS, "corrections": [],
            "source": "l1c(hist_split) + l2a + l2a_summer", "train": asdict(cfg), "split": "split_seed42",
            "pseudo_labels": {"teacher": R3_TEACHER, "selection": "view_agree", "caps": CAPS,
                              "repeat": REPEAT, "n_images": int(len(hard_idx))}})
del tr_images, tr_labels

## 4 · Comparison with E09a, E17a and E17b (all without AdaBN)

In [ ]:
rows, views_of = {}, {}
for name, run in {"E09a": REFERENCE, "E17a": "l2a_pl_r3_view_agree_contrast_aug_effb0",
                  "E17b": "l2a_summer_pl_r3_view_agree_contrast_aug_effb0"}.items():
    d = find_run(run)
    m, n, _ = load_run(d, device)
    views_of[name] = predict_views(m, data.kg_images, n)
    rows[name] = kaggle_score(d)
    del m
r = results["E18a"]
ref_pred = views_of["E09a"].mean(0).argmax(1)
summary = pd.DataFrame({k: {"Kaggle public (%)": rows[k], "confidence (%)": v.mean(0).max(1).mean() * 100,
                            "view agreement (%)": view_agreement(v) * 100,
                            "same class as E09a (%)": (v.mean(0).argmax(1) == ref_pred).mean() * 100}
                        for k, v in views_of.items()}).T
summary.loc["E18a"] = pd.Series({"Kaggle public (%)": kaggle_score(r.run_dir), "confidence (%)": r.kg_prob.max(1).mean() * 100,
                       "same class as E09a (%)": (r.kg_prob.argmax(1) == ref_pred).mean() * 100})
summary.round(2)

In [ ]:
# Different predictions: E09a's class (rows) -> E18a's class, no AdaBN (columns)
ref_names = np.array(CLASSES)[ref_pred]
new = np.array(CLASSES)[r.kg_prob.argmax(1)]
changed = new != ref_names
print(f"E18a: {changed.sum()} images differ from E09a ({changed.mean():.1%})")
pd.crosstab(ref_names[changed], new[changed]).rename_axis(index="E09a", columns="E18a")

In [ ]:
plot_variant_curves(results)

## 5 · Conclusion

*To fill in after the Kaggle submission.*

| Model | Training images | Kaggle public (no AdaBN / AdaBN) |
|---|---|---|
| E09a | L1C + hist_split | 72.67% / 71.78% |
| E17a | L2A Apr-Oct | 71.97% / 70.61% |
| E17b | L2A Jun-Aug | 72.02% / 70.89% |
| E18a | all three | |

Then: E18a in the ensemble (E15 notebook), alone or in place of E17a/E17b.